# Previsao temporal de NORM

Dados do mes `t` preveem borra com NORM no mes `t+1`. Modelo e limiar sao escolhidos na validacao temporal; o periodo final fica reservado para teste.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    average_precision_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists(): BASE_DIR = BASE_DIR.parent
PASTA_DADOS = BASE_DIR / 'data' / 'processed'
PLATAFORMA, TIPO = 'LOCAL DA GERAÇÃO', 'TIPO DE RESÍDUO'
ALVO, ALVO_FUTURO = 'TEM_NORM_MES', 'TEM_NORM_PROXIMO_MES'
QUIMICA = ['SALINIDADE', 'BARIO', 'ESTRONCIO']
PRECISAO_MINIMA = 0.50  # compromisso global entre alertas falsos e casos encontrados
PLATAFORMA_ANALISE = 'P-48'

fenix = pd.read_csv(PASTA_DADOS / 'fenix_processado.csv')
residuos = pd.read_csv(PASTA_DADOS / 'base_integrada_residuos.csv')
for df in (fenix, residuos): df['Mes'] = pd.to_datetime(df['Mes'], errors='coerce')
chaves = [PLATAFORMA, 'Mes']
tipo = residuos[TIPO].astype('string').str.strip()
eh_borra = tipo.str.contains('BORRA OLEOSA', case=False, na=False)
eh_norm = tipo.str.contains('BORRA OLEOSA COM NORM', case=False, na=False)
rotulos = residuos.loc[eh_borra, chaves].copy()
rotulos[ALVO] = eh_norm.loc[rotulos.index].astype(int)
rotulos = rotulos.groupby(chaves, as_index=False)[ALVO].max()
display(rotulos[ALVO].value_counts().rename('quantidade').to_frame())

## Variaveis e alvo futuro

Sao usadas quimica e operacao atuais, defasagem de um mes e medias dos 3 e 6 meses anteriores. O rotulo de `t+1` e ligado explicitamente a `t`, sem aproximar datas ausentes.

In [ ]:
atuais = [c for c in QUIMICA + ['QW_M3D', 'QO_M3D', 'BSW', 'N_POCOS'] if c in fenix]
dados = fenix[chaves + atuais].copy().sort_values(chaves).reset_index(drop=True)
dados[atuais] = dados[atuais].apply(pd.to_numeric, errors='coerce')
historicas = []
for coluna in QUIMICA:
    grupo = dados.groupby(PLATAFORMA, sort=False)[coluna]
    nomes = [f'{coluna}_LAG1', f'{coluna}_MEDIA_3M', f'{coluna}_MEDIA_6M']
    dados[nomes[0]] = grupo.shift(1)
    dados[nomes[1]] = grupo.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    dados[nomes[2]] = grupo.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
    historicas.extend(nomes)

rotulos_futuros = rotulos.rename(columns={ALVO: ALVO_FUTURO}).copy()
rotulos_futuros['Mes'] -= pd.offsets.MonthBegin(1)
dados = dados.merge(rotulos_futuros, on=chaves, how='left')
dados['Mes_alvo'] = dados['Mes'] + pd.offsets.MonthBegin(1)
FEATURES = atuais + historicas
quimica_valida = dados[QUIMICA].notna().all(axis=1) & (dados[QUIMICA] > 0).all(axis=1)
base = dados.loc[quimica_valida & dados[ALVO_FUTURO].notna()].sort_values('Mes').copy()
base[ALVO_FUTURO] = base[ALVO_FUTURO].astype(int)
print(f'{len(base)} observacoes, {base[PLATAFORMA].nunique()} plataformas, {len(FEATURES)} variaveis')
display(base[ALVO_FUTURO].value_counts().rename('quantidade').to_frame())

## Selecao temporal do modelo e do limiar global

Datas completas formam treino (70%), validacao (15%) e teste (15%). O modelo vence pela PR-AUC. Considerando todas as plataformas da validacao, o limiar maximiza recall enquanto exige precisao minima de 50%.

In [ ]:
datas = np.array(sorted(base['Mes'].unique()))
i70, i85 = int(len(datas)*.70)-1, int(len(datas)*.85)-1
corte_treino, corte_validacao = pd.Timestamp(datas[i70]), pd.Timestamp(datas[i85])
treino = base[base['Mes'] <= corte_treino].copy()
validacao = base[(base['Mes'] > corte_treino) & (base['Mes'] <= corte_validacao)].copy()
teste = base[base['Mes'] > corte_validacao].copy()
for nome, parte in {'Treino': treino, 'Validacao': validacao, 'Teste': teste}.items():
    print(f'{nome}: {len(parte)} linhas, {parte[PLATAFORMA].nunique()} plataformas, '          f'{parte["Mes"].min():%Y-%m} a {parte["Mes"].max():%Y-%m}')

def criar_modelos():
    return {
      'Regressao logistica': Pipeline([('imputacao', SimpleImputer(strategy='median')),
        ('escala', StandardScaler()), ('modelo', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))]),
      'Random Forest': Pipeline([('imputacao', SimpleImputer(strategy='median')),
        ('modelo', RandomForestClassifier(n_estimators=500, min_samples_leaf=5, max_features='sqrt',
         class_weight='balanced_subsample', random_state=42, n_jobs=-1))])}

comparacao, ajustados = [], {}
for nome, modelo in criar_modelos().items():
    modelo.fit(treino[FEATURES], treino[ALVO_FUTURO])
    prob = modelo.predict_proba(validacao[FEATURES])[:, 1]
    comparacao.append({'MODELO': nome, 'AUC_ROC': roc_auc_score(validacao[ALVO_FUTURO], prob),
                       'AUC_PR': average_precision_score(validacao[ALVO_FUTURO], prob)})
    ajustados[nome] = (modelo, prob)
comparacao = pd.DataFrame(comparacao).sort_values(['AUC_PR', 'AUC_ROC'], ascending=False).reset_index(drop=True)
nome_modelo = comparacao.loc[0, 'MODELO']
prob_validacao = ajustados[nome_modelo][1]

linhas = []
for limiar in np.linspace(.01, .99, 981):
    previsto = (prob_validacao >= limiar).astype(int)
    linhas.append({'LIMIAR': limiar,
      'PRECISAO': precision_score(validacao[ALVO_FUTURO], previsto, zero_division=0),
      'RECALL': recall_score(validacao[ALVO_FUTURO], previsto, zero_division=0),
      'F1': f1_score(validacao[ALVO_FUTURO], previsto, zero_division=0),
      'ACURACIA_BALANCEADA': balanced_accuracy_score(validacao[ALVO_FUTURO], previsto)})
tabela_limiares = pd.DataFrame(linhas)
candidatos = tabela_limiares[tabela_limiares['PRECISAO'] >= PRECISAO_MINIMA]
if candidatos.empty: raise ValueError('Nenhum limiar atingiu a precisao minima.')
melhor = candidatos.sort_values(['RECALL', 'F1', 'PRECISAO'], ascending=False).iloc[0]
limiar_global = float(melhor['LIMIAR'])
display(comparacao.round(3)); display(melhor.round(3).rename('validacao_global').to_frame())
print('Escolhido:', nome_modelo, '| limiar global:', round(limiar_global, 3),
      '| plataformas na validacao:', validacao[PLATAFORMA].nunique())

## Teste temporal final

In [ ]:
desenvolvimento = pd.concat([treino, validacao], ignore_index=True)
modelo_final = criar_modelos()[nome_modelo]
modelo_final.fit(desenvolvimento[FEATURES], desenvolvimento[ALVO_FUTURO])
y_teste = teste[ALVO_FUTURO]
prob_teste = modelo_final.predict_proba(teste[FEATURES])[:, 1]
prev_teste = (prob_teste >= limiar_global).astype(int)
tn, fp, fn, tp = confusion_matrix(y_teste, prev_teste, labels=[0, 1]).ravel()
metricas = pd.Series({'AUC ROC': roc_auc_score(y_teste, prob_teste),
 'AUC PR': average_precision_score(y_teste, prob_teste), 'Referencia PR': y_teste.mean(),
 'Acuracia': accuracy_score(y_teste, prev_teste),
 'Acuracia balanceada': balanced_accuracy_score(y_teste, prev_teste),
 'Precisao': precision_score(y_teste, prev_teste), 'Recall': recall_score(y_teste, prev_teste),
 'F1': f1_score(y_teste, prev_teste), 'VN': tn, 'FP': fp, 'FN': fn, 'VP': tp})
display(metricas.round(3).rename('teste_final').to_frame())
print(classification_report(y_teste, prev_teste, digits=3))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
roc = RocCurveDisplay.from_predictions(y_teste, prob_teste, name=nome_modelo, ax=axes[0])
roc.line_.set_color('tab:blue'); axes[0].plot([0,1], [0,1], '--', color='gray')
ConfusionMatrixDisplay.from_predictions(y_teste, prev_teste, display_labels=['Sem NORM','Com NORM'], cmap='Blues', ax=axes[1])
axes[1].set_title(f'Limiar global = {limiar_global:.2f}'); plt.tight_layout(); plt.show()

## Plataforma selecionada

Altere apenas `PLATAFORMA_ANALISE` no inicio. Esta avaliacao reutiliza o modelo, o limiar global e o mesmo periodo final.

In [ ]:
linha = dados.loc[dados[PLATAFORMA].eq(PLATAFORMA_ANALISE) & quimica_valida].copy()
linha['PROB_NORM'] = modelo_final.predict_proba(linha[FEATURES])[:, 1]
linha['PREVISAO_NORM'] = (linha['PROB_NORM'] >= limiar_global).astype(int)
teste_plat = linha[(linha['Mes'] > corte_validacao) & linha[ALVO_FUTURO].notna()].copy()
print('Plataforma:', PLATAFORMA_ANALISE, '| meses avaliaveis:', len(teste_plat))
if not teste_plat.empty:
    y, p = teste_plat[ALVO_FUTURO].astype(int), teste_plat['PREVISAO_NORM']
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0,1]).ravel()
    resultado = {'Acuracia': accuracy_score(y,p), 'Precisao': precision_score(y,p,zero_division=0),
      'Recall': recall_score(y,p,zero_division=0), 'F1': f1_score(y,p,zero_division=0),
      'VN':tn, 'FP':fp, 'FN':fn, 'VP':tp}
    if y.nunique()==2: resultado.update({'AUC ROC':roc_auc_score(y,teste_plat['PROB_NORM']),
      'AUC PR':average_precision_score(y,teste_plat['PROB_NORM'])})
    display(pd.Series(resultado).round(3).rename(PLATAFORMA_ANALISE).to_frame())
    display(teste_plat[['Mes_alvo',ALVO_FUTURO,'PROB_NORM','PREVISAO_NORM']].round(3))

fig, ax = plt.subplots(figsize=(14,6))
ax.plot(linha['Mes_alvo'], linha['PROB_NORM'], color='tab:blue', marker='.', label='Probabilidade prevista um mes antes')
sem, com = linha[linha[ALVO_FUTURO].eq(0)], linha[linha[ALVO_FUTURO].eq(1)]
ax.scatter(sem['Mes_alvo'], sem[ALVO_FUTURO], color='tab:green', s=45, label='Real: sem NORM')
ax.scatter(com['Mes_alvo'], com[ALVO_FUTURO], color='tab:red', s=45, label='Real: com NORM')
ax.axhline(limiar_global, color='black', linestyle='--', label=f'Limiar global = {limiar_global:.2f}')
ax.axvline(corte_validacao + pd.offsets.MonthBegin(1), color='tab:orange', linestyle='--', label='Inicio do teste')
ax.set(title=f'Previsao de NORM - {PLATAFORMA_ANALISE}', xlabel='Mes previsto', ylabel='Probabilidade', ylim=(-.05,1.05))
ax.grid(alpha=.25); ax.legend(); plt.show()

In [ ]:
pasta_modelos = BASE_DIR / 'outputs' / 'models'; pasta_modelos.mkdir(parents=True, exist_ok=True)
artefato = {'modelo':modelo_final, 'features':FEATURES, 'alvo':ALVO_FUTURO, 'horizonte_meses':1,
 'limiar':limiar_global, 'precisao_minima_validacao':PRECISAO_MINIMA, 'modelo_selecionado':nome_modelo,
 'fim_treino':corte_treino, 'fim_validacao':corte_validacao}
caminho = pasta_modelos / 'modelo_norm_proximo_mes.joblib'
joblib.dump(artefato, caminho); print('Modelo salvo em:', caminho)